In [ ]:
!git clone -b audio_flamingo_2 https://github.com/NVIDIA/audio-flamingo.git
%cd audio-flamingo/inference_HF_pretrained
import os
print(os.listdir("."))  # проверяем что inference.py на месте

Cloning into 'audio-flamingo'...
remote: Enumerating objects: 1173, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 1173 (delta 122), reused 106 (delta 104), pack-reused 1022 (from 1)
Receiving objects: 100% (1173/1173), 21.92 MiB | 18.58 MiB/s, done.
Resolving deltas: 100% (610/610), done.
Filtering content: 100% (2/2), 1.37 MiB | 1.03 MiB/s, done.
/content/audio-flamingo/inference_HF_pretrained
['utils.py', 'inference_CoT.py', 'configs', 'inference_CoT.jsonl', 'requirements.txt', '.gitignore', '.gitattributes', 'my_laion_clap', 'inference.jsonl', 'README.md', 'inference.py', 'src']


In [ ]:
!pip install -q torch torchaudio transformers accelerate einops soundfile librosa torchlibrosa ftfy braceexpand webdataset wget einops_exts flamingo

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 117.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

inference_py = Path("/content/audio-flamingo/inference_HF_pretrained/inference.py")
factory_py = Path("/content/audio-flamingo/inference_HF_pretrained/src/factory.py")
config_yaml = Path("/content/audio-flamingo/inference_HF_pretrained/configs/inference.yaml")

# ======= ПАТЧ 1: factory.py — weights_only=False для torch.load =======
factory_text = factory_py.read_text()
factory_patched = factory_text.replace(
    'torch.load(clap_config["checkpoint"], map_location = \'cpu\')',
    'torch.load(clap_config["checkpoint"], map_location=\'cpu\', weights_only=False)'
)
if factory_patched != factory_text:
    factory_py.write_text(factory_patched)
    print("✅ Патч 1 применён: weights_only=False")
else:
    print("⚠️ Патч 1: строка не найдена или уже пропатчено")

# ======= ПАТЧ 2: inference.py — HF_TOKEN =======
from google.colab import userdata
HF_TOKEN = userdata.get('HF')

inference_text = inference_py.read_text()
inference_patched = inference_text.replace(
    'YOUR_HF_TOKEN',
    HF_TOKEN
).replace('"nvidia/audio-flamingo-2"', '"nvidia/audio-flamingo-2-1.5B"')
if inference_patched != inference_text:
    inference_text = inference_patched
    inference_py.write_text(inference_text)
    print("✅ Патч 2 применён: HF_TOKEN")
else:
    print("⚠️ Патч 2: уже пропатчено или строка не найдена")

# ======= ПАТЧ 3: inference.yaml — переключение на Qwen2.5-1.5B =======
config_text = config_yaml.read_text()

config_patched = config_text.replace(
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-1.5B"
).replace(
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-1.5B"
)

if config_patched != config_text:
    config_yaml.write_text(config_patched)
    print("✅ Патч 2 применён: inference.yaml -> Qwen/Qwen2.5-1.5B")
else:
    print("⚠️ Патч 2: строка Qwen/Qwen2.5-3B не найдена или уже пропатчено")

⚠️ Патч 1: строка не найдена или уже пропатчено
⚠️ Патч 2: уже пропатчено или строка не найдена
⚠️ Патч 2: строка Qwen/Qwen2.5-3B не найдена или уже пропатчено


In [ ]:
import numpy as np
import torch.serialization


# Регистрируем numpy-глобалы как безопасные прямо в текущей сессии
torch.serialization.add_safe_globals([
    np.core.multiarray.scalar,
    np.dtype,
    np.ndarray,
])

/tmp/ipykernel_2825/3107404843.py:7: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray.scalar,


In [ ]:
import tempfile, json, pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/data.parquet")
row = df.iloc[0]

audio_bytes = row['audio']['bytes']
suffix = Path(row['audio'].get('path', 'track.mp3')).suffix or '.mp3'
tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
tmp.write(audio_bytes)
tmp.close()
audio_path = tmp.name
print("Аудиофайл:", audio_path)

QUESTION = "Describe this music in detail: genre, mood, instruments, tempo, structure."

input_data = {
    "path": audio_path,       # было "audio" — нужно "path"
    "prompt": QUESTION        # было "question" — нужно "prompt"
}

with open("/content/audio-flamingo/inference_HF_pretrained/inference.jsonl", "w") as f:
    f.write(json.dumps(input_data) + "\n")

print("inference.jsonl готов.")

Mounted at /content/drive
Аудиофайл: /tmp/tmpmdmunfvk.mp3
inference.jsonl готов.


In [ ]:
%cd /content/audio-flamingo/inference_HF_pretrained

import os


assert HF_TOKEN, "❌ HF_TOKEN пустой"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

!python inference.py --input inference.jsonl

/content/audio-flamingo/inference_HF_pretrained
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
tokenizer_config.json: 100% 

In [ ]:
QUESTION = (
    "You are a listener trying to describe this track. Listen to the audio track "
    "and write one coherent paragraph"
    "In 5 natural sentences, cover in order: (1) atmosphere and mood"
    "(2) general style or genre direction in simple, everyday words"
    "(3) energy"
    "(4) dominant instruments or sound textures, and (5) vocals if present and how they come across. "
    "Use varied, natural wording"
    "Do not invent exact technical values like BPM, key, or time signature, and do not use labels, lists, "
    "metadata fields, code-like text, or specialized technical terms."
)


# Copyright (c) 2025 NVIDIA CORPORATION.
#   Licensed under the MIT license.

import os
import yaml
import json
import argparse

import torch
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

from src.factory import create_model_and_transforms
from utils import Dict2Class, get_autocast, get_cast_dtype

def int16_to_float32(x):
    return (x / 32767.0).astype(np.float32)

def float32_to_int16(x):
    x = np.clip(x, a_min=-1., a_max=1.)
    return (x * 32767.).astype(np.int16)

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def get_num_windows(T, sr, clap_config):

    window_length  = int(float(clap_config["window_length"]) * sr)
    window_overlap = int(float(clap_config["window_overlap"]) * sr)
    max_num_window = int(clap_config["max_num_window"])

    num_windows = 1
    if T <= window_length:
        num_windows = 1
        full_length = window_length
    elif T >= (max_num_window * window_length - (max_num_window - 1) * window_overlap):
        num_windows = max_num_window
        full_length = (max_num_window * window_length - (max_num_window - 1) * window_overlap)
    else:
        num_windows = 1 + int(np.ceil((T - window_length) / float(window_length - window_overlap)))
        full_length = num_windows * window_length - (num_windows - 1) * window_overlap

    return num_windows, full_length


def read_audio(file_path, target_sr, duration, start, clap_config):

    if file_path.endswith('.mp3'):
        audio = AudioSegment.from_file(file_path)
        if len(audio) > (start + duration) * 1000:
            audio = audio[start * 1000:(start + duration) * 1000]

        if audio.frame_rate != target_sr:
            audio = audio.set_frame_rate(target_sr)

        if audio.channels > 1:
            audio = audio.set_channels(1)

        data = np.array(audio.get_array_of_samples())
        if audio.sample_width == 2:
            data = data.astype(np.float32) / np.iinfo(np.int16).max
        elif audio.sample_width == 4:
            data = data.astype(np.float32) / np.iinfo(np.int32).max
        else:
            raise ValueError("Unsupported bit depth: {}".format(audio.sample_width))

    else:
        with sf.SoundFile(file_path) as audio:
            original_sr = audio.samplerate
            channels = audio.channels

            max_frames = int((start + duration) * original_sr)

            audio.seek(int(start * original_sr))
            frames_to_read = min(max_frames, len(audio))
            data = audio.read(frames_to_read)

            if data.max() > 1 or data.min() < -1:
                data = data / max(abs(data.max()), abs(data.min()))

        if original_sr != target_sr:
            if channels == 1:
                data = librosa.resample(data.flatten(), orig_sr=original_sr, target_sr=target_sr)
            else:
                data = librosa.resample(data.T, orig_sr=original_sr, target_sr=target_sr)[0]
        else:
            if channels != 1:
                data = data.T[0]

    if data.min() >= 0:
        data = 2 * data / abs(data.max()) - 1.0
    else:
        data = data / max(abs(data.max()), abs(data.min()))

    assert len(data.shape) == 1, data.shape
    return data

def load_audio(audio_path, clap_config):

    sr = 16000
    window_length  = int(float(clap_config["window_length"]) * sr)
    window_overlap = int(float(clap_config["window_overlap"]) * sr)
    max_num_window = int(clap_config["max_num_window"])
    duration = max_num_window * (clap_config["window_length"] - clap_config["window_overlap"]) + clap_config["window_overlap"]

    audio_data = read_audio(audio_path, sr, duration, 0.0, clap_config) # hard code audio start to 0.0
    T = len(audio_data)
    num_windows, full_length = get_num_windows(T, sr, clap_config)

    # pads to the nearest multiple of window_length
    if full_length > T:
        audio_data = np.append(audio_data, np.zeros(full_length - T))

    audio_data = audio_data.reshape(1, -1)
    audio_data_tensor = torch.from_numpy(int16_to_float32(float32_to_int16(audio_data))).float()

    audio_clips = []
    audio_embed_mask = torch.ones(num_windows)
    for i in range(num_windows):
        start = i * (window_length - window_overlap)
        audio_data_tensor_this = audio_data_tensor[:, start:start+window_length]
        audio_clips.append(audio_data_tensor_this)

    if len(audio_clips) > max_num_window:
        audio_clips = audio_clips[:max_num_window]
        audio_embed_mask = audio_embed_mask[:max_num_window]

    audio_clips = torch.cat(audio_clips)

    return audio_clips, audio_embed_mask

def predict(filepath, question, clap_config, inference_kwargs):

    audio_clips, audio_embed_mask = load_audio(filepath, clap_config)

    # ВАЖНО: не переводим вручную в bf16/fp16, оставляем float32
    audio_clips = audio_clips.to(device_id, dtype=torch.float32, non_blocking=True)
    audio_embed_mask = audio_embed_mask.to(device_id, dtype=torch.float32, non_blocking=True)

    text_prompt = str(question).lower()

    sample = f"<audio>{text_prompt.strip()}{tokenizer.sep_token}"

    text = tokenizer(
        sample,
        max_length=512,
        padding="longest",
        truncation="only_first",
        return_tensors="pt"
    )

    input_ids = text["input_ids"].to(device_id, non_blocking=True)
    attention_mask = text["attention_mask"].to(device_id, non_blocking=True)

    prompt = input_ids

    with torch.no_grad():
        output = model.generate(
            audio_x=audio_clips.unsqueeze(0),
            audio_x_mask=audio_embed_mask.unsqueeze(0),
            lang_x=prompt,
            attention_mask=attention_mask,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=256,
            **inference_kwargs,
        )[0]

    output_decoded = tokenizer.decode(output, skip_special_tokens=False)
    output_decoded = output_decoded.split(tokenizer.sep_token)[-1]
    output_decoded = output_decoded.replace(tokenizer.eos_token, '')
    output_decoded = output_decoded.replace(tokenizer.pad_token, '')
    output_decoded = output_decoded.replace('<|endofchunk|>', '')
    output_decoded = output_decoded.strip()

    print('Prompt:', question)
    print('Audio Flamingo 2:', output_decoded)

    return output_decoded


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--input", "-i", type=str, help="Path to input JSON file")
    parsed_args = parser.parse_args()

    snapshot_download(repo_id="nvidia/audio-flamingo-2-1.5B", local_dir="./", token="")

    config = yaml.load(open("configs/inference.yaml"), Loader=yaml.FullLoader)

    data_config = config['data_config']
    model_config = config['model_config']
    clap_config = config['clap_config']
    args = Dict2Class(config['train_config'])

    # Для инференса форсируем fp32
    args.precision = "32"

    model, tokenizer = create_model_and_transforms(
        **model_config,
        clap_config=clap_config,
        use_local_files=args.offline,
        gradient_checkpointing=args.gradient_checkpointing,
        freeze_lm_embeddings=args.freeze_lm_embeddings,
    )

    device_id = 0
    model = model.to(device_id)
    model.eval()
    model = model.to(torch.float32)
    # Load metadata
    with open("safe_ckpt/metadata.json", "r") as f:
        metadata = json.load(f)

    # Reconstruct the full state_dict
    state_dict = {}

    # Load each SafeTensors chunk
    for chunk_name in metadata:
        chunk_path = f"safe_ckpt/{chunk_name}.safetensors"
        chunk_tensors = load_file(chunk_path)

        # Merge tensors into state_dict
        state_dict.update(chunk_tensors)

    missing_keys, unexpected_keys = model.load_state_dict(state_dict, False)

    '''autocast = get_autocast(
        args.precision, cache_enabled=(not args.fsdp)
    )

    cast_dtype = get_cast_dtype(args.precision)'''

    data = []
    with open(parsed_args.input, "r", encoding="utf-8") as file:
        for line in file:
            data.append(json.loads(line.strip()))

    inference_kwargs = {
        "do_sample": True,
        "top_k": 30,
        "top_p": 0.95,
        "num_return_sequences": 1
    }

    results = []

    for i, item in enumerate(data, start=1):
        print(f"[{i}/{len(data)}] Processing: {item['track_name']}")

        text = predict(
            item["path"],
            QUESTION,
            clap_config,
            inference_kwargs
        )

        results.append({
            "idx": item["idx"],
            "track_name": item["track_name"],
            "description": text
        })

    with open("predictions.jsonl", "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nSaved {len(results)} predictions to predictions.jsonl")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):

KeyboardInterrupt



In [ ]:
# Старый рабочий вариант
import tempfile
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

df = pd.read_parquet("/content/drive/MyDrive/data.parquet")

# Берём ровно 500 случайных строк.
# random_state фиксирует выбор для воспроизводимости.
df_sample = df.sample(n=500, random_state=42)

QUESTION = (
    "You are an expert music critic. "
    "Your task is to listen to an audio track and write a vivid description of how the music sounds and feels. "
    "In 3–4 sentences, describe the atmosphere and mood, the general musical style, the rhythm and energy, "
    "the main instruments and sounds you hear, and, if there is a voice, how the vocal performance and emotion sound. "
    "Do not use technical or programming terms (no function_call, parameters, code, JSON, key, IDs, file formats). "
    "Do not talk to the listener directly and do not mention playlists, apps, or recommendations. "
    "Output exactly one paragraph of 2–3 sentences in natural, expressive language, "
    "like a short review in a music magazine. Write only the description of the music."
)

out_path = "/content/audio-flamingo/inference_HF_pretrained/inference.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        audio_bytes = row["audio"]["bytes"]

        original_path = row["audio"].get("path", f"track_{idx}.mp3")
        suffix = Path(original_path).suffix or ".mp3"

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.write(audio_bytes)
        tmp.close()

        rec = {
            "idx": int(idx),          # индекс в исходном df (полезно, если захочешь матчинг)
            "track_name": original_path,
            "path": tmp.name,
            "prompt": QUESTION,
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Подготовлено {len(df_sample)} треков")

  0%|          | 0/500 [00:00<?, ?it/s]

Подготовлено 500 треков


In [ ]:
# Создание jsonl без промта
import tempfile
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

df = pd.read_parquet("/content/drive/MyDrive/data.parquet")

# Берём 500 случайных треков
df_sample = df.sample(n=500, random_state=42)

out_path = "/content/audio-flamingo/inference_HF_pretrained/inference.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        audio_bytes = row["audio"]["bytes"]
        original_path = row["audio"].get("path", f"track_{idx}.mp3")
        suffix = Path(original_path).suffix or ".mp3"

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.write(audio_bytes)
        tmp.close()

        rec = {
            "idx": int(idx),           # индекс строки в исходном df
            "track_name": original_path,
            "path": tmp.name
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Готово, подготовлено 500 треков")

  0%|          | 0/500 [00:00<?, ?it/s]

Готово, подготовлено 500 треков


In [ ]:
%cd /content/audio-flamingo/inference_HF_pretrained

import os


assert HF_TOKEN, "❌ HF_TOKEN пустой"
os.environ["HF"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

!python inference.py --input inference.jsonl

/content/audio-flamingo/inference_HF_pretrained
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 238kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 996kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.73MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 142kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 24.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 967kB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 3.00MB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 16.0MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 13.0MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 14.4MB/s]
/content/audio-flamingo/inference_HF_pretrained/src/helpers.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. P

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pred_df = pd.read_json(
    "/content/audio-flamingo/inference_HF_pretrained/predictions.jsonl",
    lines=True
)

df["description"] = None

df.loc[
    pred_df["idx"],
    "description"
] = pred_df["description"].values

In [ ]:
output_path = "/content/drive/MyDrive/data_with_descriptions_best_prompt.parquet"

df.to_parquet(output_path, index=False)

print("Сохранено:", output_path)

Сохранено: /content/drive/MyDrive/data_with_descriptions_best_prompt.parquet
